<a href="https://colab.research.google.com/github/Anamika-Thakur/Sentiment-Analysis/blob/main/Sentiment%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [ ]:
import os
import re
import urllib.request
import tarfile
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Unzip file

In [37]:
import tarfile
import zipfile
import os

archive_name = None
for file in os.listdir('.'):
    if file.endswith('.tar.gz') or file.endswith('.zip'):
        archive_name = file
        break

if archive_name.endswith('.zip'):
    with zipfile.ZipFile(archive_name, 'r') as zip_ref:
        zip_ref.extractall('.')
else:
    with tarfile.open(archive_name, 'r:gz') as tar_ref:
        tar_ref.extractall('.')

print("Extraction complete!\n")

Detected archive: 'aclImdb_v1.tar.gz'. Extracting...


/tmp/ipykernel_12455/3432520034.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall('.')


Extraction complete!



file of interest

In [39]:
def load_reviews_from_split(split_type):
    reviews = []
    labels = []

    for sentiment in ['pos', 'neg']:
        dir_path = os.path.join(base_dir, split_type, sentiment)
        label = 1 if sentiment == 'pos' else 0

        for file_name in os.listdir(dir_path):
            # Explicitly only read individual review text files
            if file_name.endswith('.txt'):
                file_path = os.path.join(dir_path, file_name)
                with open(file_path, 'r', encoding='utf-8') as f:
                    reviews.append(f.read())
                    labels.append(label)
    return reviews, labels


Train Data

In [41]:
base_dir = 'aclImdb'
print("Loading training data (ignoring unsupervised files)...")
train_reviews, train_labels = load_reviews_from_split('train')
print(f"-> Loaded {len(train_reviews)} training reviews.")

Loading training data (ignoring unsupervised files)...
-> Loaded 25000 training reviews.


Test Data

In [42]:
print("Loading testing data...")
test_reviews, test_labels = load_reviews_from_split('test')
print(f"-> Loaded {len(test_reviews)} testing reviews.\n")

Loading testing data...
-> Loaded 25000 testing reviews.



## Text processing and cleaning

In [44]:
import re

def clean_text(text):
    # Remove HTML break tags (<br />, etc.)
    text = re.sub(r'<br\s*/?>', ' ', text)
    # Strip out special characters, punctuation, and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase and compress whitespace
    text = text.lower()
    return re.sub(r'\s+', ' ', text).strip()

print("Cleaning and normalising text data...")
train_reviews_clean = [clean_text(r) for r in train_reviews]
test_reviews_clean = [clean_text(r) for r in test_reviews]
print("Text cleaning complete!\n")

Cleaning and normalising text data...
Text cleaning complete!



## Feature Extraction (TF-IDF)

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("Vectorising text features using TF-IDF (Unigrams + Bigrams)...")
# Setting dual=False explicitly later requires a clean feature matrix size
vectorizer = TfidfVectorizer(max_features=25000, stop_words='english', ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_reviews_clean)
X_test = vectorizer.transform(test_reviews_clean)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}\n")

Vectorising text features using TF-IDF (Unigrams + Bigrams)...
Train Shape: (25000, 25000) | Test Shape: (25000, 25000)



## SVM Training

In [49]:
from sklearn.svm import LinearSVC


# dual=False is optimal for problems where n_samples > n_features
svm = LinearSVC(C=1.0, dual=False, random_state=42, max_iter=2000)
svm.fit(X_train, y_train)
print("Training complete!\n")

Training complete!



## Evaluation

In [50]:
# Model Predictions
y_pred = svm.predict(X_test)

# Report results
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Accuracy Score: 87.19%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.88      0.87     12500
    Positive       0.88      0.86      0.87     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87      0.87     25000
weighted avg       0.87      0.87      0.87     25000

